# Editing-cycle simulation — full pipeline

Replaces the human reviewer with an AI judge:

1. **preedit** — read existing robot predictions, parse top-1 *(no GPU)*
2. **judge** — stronger model reviews from image *(Azure API, no GPU)*
3. **postedit** — robot re-diagnoses given feedback *(GPU)*
4. **eval** — pre/judge/post accuracy vs MIDAS ground truth (y16)

Run on a **GPU node** (phase 3 needs it). All outputs go to `prelim_simedit/results_local/`.

In [ ]:
import os, sys

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"
SIMEDIT = os.path.join(PROJECT_ROOT, "prelim_simedit")
if SIMEDIT not in sys.path:
    sys.path.insert(0, SIMEDIT)

from utils.pipeline import run_preedit, run_judge, run_postedit, run_eval
from utils.io import set_output_dir

# --- Configuration ---
ROBOT = "medgemma"         # candidates: medgemma, dermato_llama
JUDGE = "claude_opus48"            # judges: gpt54, claude_opus48, ground_truth
SEED = 42                  # for reproducibility (used wherever randomness is needed)
DIFFERENTIAL = "top_1"     # "top_1" = single diagnosis, "top_3" = top-3 differential

# Filtering (pick ONE approach):
N = 5                      # first N cases (set None for ALL)
CASE_IDS = None            # OR explicit list: ["1_combined", "5_combined", "10_combined"]

# Notebook test outputs go to results_local/test/ (can overwrite freely).
# Set to None to write to results_local/<robot>/ (official runs).
set_output_dir(os.path.join(SIMEDIT, "results_local", "test"))

In [2]:
# --- Phase 1: preedit (no GPU) ---
# Reads existing predictions CSV, parses diagnosis.
# Takes first N cases (sorted by case_id). Set CASE_IDS to override.

df_preedit = run_preedit(ROBOT, n=N, seed=SEED, case_ids=CASE_IDS,
                         differential=DIFFERENTIAL)
df_preedit[["case_id", "gt_y16", "preedit_dx"]]

build_inputs(medgemma): 5 combined cases
[preedit/medgemma/top_1] 5 cases -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/01_preedit__top_1.csv


,case_id,gt_y16,preedit_dx
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma
1,2_combined,Melanocytic Nevus,Basal Cell Carcinoma
2,6_combined,Squamous Cell Carcinoma,Basal Cell Carcinoma
3,8_combined,Other,Actinic Keratosis
4,9_combined,Seborrheic Keratosis,Basal Cell Carcinoma


In [3]:
# --- Phase 2: judge (API, no GPU) ---
# Judge sees the image + robot's diagnosis, provides its own.
# Resumable: skips case_ids already in the judge CSV.

df_judge = run_judge(robot_name=ROBOT, judge=JUDGE, differential=DIFFERENTIAL)
df_judge[[c for c in df_judge.columns if c.startswith("judge_") or c in ("case_id", "gt_y16")]]

[judge/medgemma/claude_opus48/top_1] 5/5 pending


100%|█████████████████████████████████████████████████████████████████| 5/5 [00:26<00:00,  5.36s/it]

[judge/medgemma/claude_opus48/top_1] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/02_judge__claude_opus48__top_1.csv


,case_id,gt_y16,judge_dx,judge_reasoning
0,1_combined,Squamous Cell Carcinoma In Situ,Basal cell carcinoma,Dermoscopy shows fine arborizing (branching) t...
1,2_combined,Melanocytic Nevus,Solar lentigo / lentigo (benign pigmented macule),"Dermoscopy shows a small, uniformly tan-brown,..."
2,6_combined,Squamous Cell Carcinoma,Squamous cell carcinoma (invasive/keratoacanth...,On a sun-damaged forearm of an elderly patient...
3,8_combined,Other,Actinic keratosis,"The dorsal hand of an elderly, severely sun-da..."
4,9_combined,Seborrheic Keratosis,Squamous cell carcinoma (invasive SCC arising ...,"On a sun-damaged, bald/thinning scalp of an el..."


In [4]:
# --- Phase 3: postedit (GPU) ---
# Robot re-diagnoses given the judge's feedback.
# Resumable: skips case_ids already in the postedit CSV.

df_postedit = run_postedit(ROBOT, judge_name=JUDGE, differential=DIFFERENTIAL)
if DIFFERENTIAL == "top_1":
    df_postedit[["case_id", "gt_y16", "preedit_dx", "judge_dx", "postedit_dx"]]
else:
    df_postedit[["case_id", "gt_y16", "postedit_dx"]]

[postedit/medgemma/claude_opus48/top_1] 5/5 pending


/home/jq2uw/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████████████████████████| 2/2 [00:03<00:00,  1.59s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Total params: 4,300,079,472


100%|█████████████████████████████████████████████████████████████████| 5/5 [00:11<00:00,  2.27s/it]

[postedit/medgemma/claude_opus48/top_1] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/03_postedit__claude_opus48__top_1.csv


In [5]:
# --- Phase 4: eval (no GPU) ---
# Score all phases against ground truth (y16).

scored, summary = run_eval(robot_name=ROBOT, judge_name=JUDGE,
                           differential=DIFFERENTIAL)

[eval/medgemma/claude_opus48/top_1] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/scored__claude_opus48__top_1.csv
   robot         judge differential  n  preedit_acc  judge_acc  postedit_acc  delta  n_improved  n_regressed
medgemma claude_opus48        top_1  5          0.0        0.2           0.2    0.2           1            0


In [6]:
# --- Per-case details ---
if DIFFERENTIAL == "top_1":
    df = scored[["case_id", "gt_y16",
            "preedit_dx", "preedit_y16", "preedit_correct",
            "judge_dx", "judge_y16", "judge_correct",
            "postedit_dx", "postedit_y16", "postedit_correct"]]
else:
    df = scored[["case_id", "gt_y16",
            "preedit_dx1", "preedit_top1_correct", "preedit_top3_correct",
            "judge_dx1", "judge_top1_correct", "judge_top3_correct",
            "postedit_dx1", "postedit_top1_correct", "postedit_top3_correct"]]

df

,case_id,gt_y16,preedit_dx,preedit_y16,preedit_correct,judge_dx,judge_y16,judge_correct,postedit_dx,postedit_y16,postedit_correct
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma,Basal Cell Carcinoma,False,Basal cell carcinoma,Basal Cell Carcinoma,False,Basal cell carcinoma,Basal Cell Carcinoma,False
1,2_combined,Melanocytic Nevus,Basal Cell Carcinoma,Basal Cell Carcinoma,False,Solar lentigo / lentigo (benign pigmented macule),Other Benign,False,Solar lentigo / lentigo (benign pigmented macule),Other Benign,False
2,6_combined,Squamous Cell Carcinoma,Basal Cell Carcinoma,Basal Cell Carcinoma,False,Squamous cell carcinoma (invasive/keratoacanth...,Squamous Cell Carcinoma,True,Squamous cell carcinoma (invasive/keratoacanth...,Squamous Cell Carcinoma,True
3,8_combined,Other,Actinic Keratosis,Actinic Keratosis,False,Actinic keratosis,Actinic Keratosis,False,Actinic keratosis,Actinic Keratosis,False
4,9_combined,Seborrheic Keratosis,Basal Cell Carcinoma,Basal Cell Carcinoma,False,Squamous cell carcinoma (invasive SCC arising ...,Squamous Cell Carcinoma,False,Squamous cell carcinoma (invasive SCC arising ...,Squamous Cell Carcinoma,False
